#TASK 5 — Feature Engineering

##Imorting Libraries

In [4]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

##Loading Dataset

In [5]:
df = pd.read_csv('/content/patient_feedback_cleaned.csv')
print(df.head())

           Theme                                        Feedback  Sentiment  \
0      discharge          discharge instructions were very clear          1   
1  communication  i felt the doctor used too much medical jargon          1   
2     medication  i appreciated the reminders about my medicines          1   
3      discharge   instructions after discharge were not helpful          0   
4      discharge   instructions after discharge were not helpful          0   

   Satisfaction  Readmission  
0             4            1  
1             5            0  
2             4            0  
3             1            1  
4             2            1  


### Feature Engineering: Feedback Length and Word Count

In [6]:
# Calculate Feedback Length
df['Feedback_Length'] = df['Feedback'].apply(len)

# Calculate Word Count
df['Word_Count'] = df['Feedback'].apply(lambda x: len(x.split()))

print("DataFrame with Feedback_Length and Word_Count:")
print(df[['Feedback', 'Feedback_Length', 'Word_Count']].head())

DataFrame with Feedback_Length and Word_Count:
                                         Feedback  Feedback_Length  Word_Count
0          discharge instructions were very clear               38           5
1  i felt the doctor used too much medical jargon               46           9
2  i appreciated the reminders about my medicines               46           7
3   instructions after discharge were not helpful               45           6
4   instructions after discharge were not helpful               45           6


### Feature Engineering: Sentiment Ratio by Theme

In [7]:
# Calculate the average sentiment for each theme
sentiment_ratio_by_theme = df.groupby('Theme')['Sentiment'].mean().reset_index()
sentiment_ratio_by_theme.rename(columns={'Sentiment': 'Theme_Sentiment_Ratio'}, inplace=True)

# Merge this back to the original DataFrame
df = pd.merge(df, sentiment_ratio_by_theme, on='Theme', how='left')

print("DataFrame with Theme_Sentiment_Ratio:")
print(df[['Theme', 'Sentiment', 'Theme_Sentiment_Ratio']].head())

DataFrame with Theme_Sentiment_Ratio:
           Theme  Sentiment  Theme_Sentiment_Ratio
0      discharge          1               0.500000
1  communication          1               0.714286
2     medication          1               0.500000
3      discharge          0               0.500000
4      discharge          0               0.500000


### Feature Engineering: TF-IDF Features

In [8]:
# Initialize TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=100) # Limiting to 100 features for simplicity

# Fit and transform the 'Feedback' text
tfidf_features = tfidf_vectorizer.fit_transform(df['Feedback'])

# Convert to DataFrame
tfidf_df = pd.DataFrame(tfidf_features.toarray(), columns=tfidf_vectorizer.get_feature_names_out())

# Concatenate TF-IDF features with the original DataFrame
df = pd.concat([df, tfidf_df], axis=1)

print("DataFrame with TF-IDF features (first 5 rows and selected columns):")
print(df[tfidf_vectorizer.get_feature_names_out()].head())

DataFrame with TF-IDF features (first 5 rows and selected columns):
      about  admitted     after  and  appointments  appreciated  before  \
0  0.000000       0.0  0.000000  0.0           0.0     0.000000     0.0   
1  0.000000       0.0  0.000000  0.0           0.0     0.000000     0.0   
2  0.407741       0.0  0.000000  0.0           0.0     0.478919     0.0   
3  0.000000       0.0  0.497347  0.0           0.0     0.000000     0.0   
4  0.000000       0.0  0.497347  0.0           0.0     0.000000     0.0   

   being   by     clear  ...  took  unclear  understand      used     very  \
0    0.0  0.0  0.591347  ...   0.0      0.0         0.0  0.000000  0.50346   
1    0.0  0.0  0.000000  ...   0.0      0.0         0.0  0.401089  0.00000   
2    0.0  0.0  0.000000  ...   0.0      0.0         0.0  0.000000  0.00000   
3    0.0  0.0  0.000000  ...   0.0      0.0         0.0  0.000000  0.00000   
4    0.0  0.0  0.000000  ...   0.0      0.0         0.0  0.000000  0.00000   

   wait  wai

### Feature Engineering: One-Hot Encoding for 'Theme'

In [9]:
# Perform one-hot encoding on the 'Theme' column
theme_one_hot = pd.get_dummies(df['Theme'], prefix='Theme')

# Concatenate the one-hot encoded DataFrame with the original DataFrame
df = pd.concat([df, theme_one_hot], axis=1)

print("DataFrame with One-Hot Encoded 'Theme' features (first 5 rows and selected columns):")
print(df[['Theme'] + list(theme_one_hot.columns)].head())

DataFrame with One-Hot Encoded 'Theme' features (first 5 rows and selected columns):
           Theme  Theme_communication  Theme_discharge  Theme_medication  \
0      discharge                False             True             False   
1  communication                 True            False             False   
2     medication                False            False              True   
3      discharge                False             True             False   
4      discharge                False             True             False   

   Theme_wait_time  
0            False  
1            False  
2            False  
3            False  
4            False  


### Saving Featured Dataset

In [10]:
# Save the featured DataFrame to a new CSV file
df.to_csv('patient_feedback_featured.csv', index=False)

print("Featured dataset saved to 'patient_feedback_featured.csv'")

Featured dataset saved to 'patient_feedback_featured.csv'
